In [2]:
!pip install bitsandbytes
!pip install unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3

In [8]:
# Helpful functions used through the entire notebook
import torch
import torch.nn as nn
from transformers import set_seed
import time
import inspect
import os
major_version, minor_version = torch.cuda.get_device_capability()
HAS_BFLOAT16 = (major_version >= 8)
from inspect import currentframe as _C, getframeinfo
_F = lambda c: getframeinfo(c).lineno # Gets line number
WARN = lambda x: print(f"\033[31m{x}\033[0m") # Red colored warnings

# https://stackoverflow.com/questions/18425225/getting-the-name-of-a-variable-as-a-string
def NAME(var):
    callers_local_vars = inspect.currentframe().f_back.f_locals.items()
    names = [var_name for var_name, var_val in callers_local_vars if var_val is var]
    return names[0] if len(names) != 0 else ""

def assert_same(x, y, line, dtype):
    assert(x.dtype == dtype)
    try: torch.testing.assert_close(x, y, check_stride = True)
    except Exception as error:
        raise RuntimeError(
            f"Failed allclose at line [{line}]: {NAME(x)}, {NAME(y)}\n{str(error)}"
        )

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [3]:
from bitsandbytes.nn import Linear4bit
from transformers.activations import ACT2FN
from unsloth.kernels.utils import fast_dequantize
from peft.utils.integrations import dequantize_module_weight as peft_dequantize
def unsloth_dequantize(weight):
    return fast_dequantize(weight.weight, weight.weight.quant_state)

def bnb_Linear4bit(hd, m, dtype = torch.float16):
    return Linear4bit(
        hd, m, bias = None,
        compute_dtype       = dtype,
        compress_statistics = True,
        quant_type          = "nf4",
    )

# [NEW] as at 18th Feb 2025
def assert_correct_bnb(weight, dtype):
    assert(weight.weight.dtype == torch.uint8)
    assert(weight.weight.quant_state.dtype == dtype)
    assert(weight.weight.quant_state.absmax.dtype == torch.uint8)
    assert(weight.weight.quant_state.code.dtype == torch.float32)
    assert(weight.weight.quant_state.offset.dtype == torch.float32)
    assert(weight.weight.quant_state.blocksize == 64)
    assert(weight.weight.quant_state.state2.absmax.dtype == torch.float32)
    assert(weight.weight.quant_state.state2.code.dtype == torch.float32)
    assert(weight.weight.quant_state.state2.blocksize == 256)

class MLP(nn.Module):
    def __init__(self, hd = 4096, m = 14336, dtype = torch.float16):
        super().__init__()
        self.gate_proj = bnb_Linear4bit(hd, m, dtype = dtype).to("cuda")
        self.up_proj   = bnb_Linear4bit(hd, m, dtype = dtype).to("cuda")
        self.down_proj = bnb_Linear4bit(m, hd, dtype = dtype).to("cuda")
        # [NEW] as at 18th Feb 2025
        self.gate_proj.weight.quant_state.dtype = dtype
        self.up_proj  .weight.quant_state.dtype = dtype
        self.down_proj.weight.quant_state.dtype = dtype
        self.act_fn = ACT2FN["silu"]
    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

def mlp_forward(X, mlp, fx):
    up   = X @ fx(mlp.  up_proj).t()
    gate = X @ fx(mlp.gate_proj).t()
    h = mlp.act_fn(gate) * up
    down = h @ fx(mlp.down_proj).t()
    return down

def mlp_dequantize(X, mlp, fx):
    a = fx(mlp.  up_proj).t(); torch.cuda.synchronize()
    b = fx(mlp.gate_proj).t(); torch.cuda.synchronize()
    c = fx(mlp.down_proj).t(); torch.cuda.synchronize()
    return a, b, c

def test_dequantize(dequantize_fx):
    elapsed = 0
    options = [
        (2, 3333, 2048,  8192, 3407, torch.float16),
        (5,  777, 1024,  4096, 3409, torch.bfloat16),
        (3, 2048, 4096, 14336, 3408, torch.bfloat16),
    ]
    for (bsz, qlen, hd, m, seed, dt) in options:
        set_seed(seed)
        torch.set_default_dtype(torch.float32)
        mlp = MLP(hd = hd, m = m, dtype = dt)
        X = torch.randn((bsz, qlen, hd), device = "cuda", dtype = dt)
        torch.cuda.synchronize()

        # Warmup
        for _ in range(2):
            assert_same( mlp_forward(X, mlp, dequantize_fx), mlp(X), _F(_C()), dt)
            # [NEW] as at 18th Feb 2025
            assert_correct_bnb(mlp.  up_proj, dt)
            assert_correct_bnb(mlp.gate_proj, dt)
            assert_correct_bnb(mlp.down_proj, dt)
            a, b, c = mlp_dequantize(X, mlp, dequantize_fx)
            A, B, C = mlp_dequantize(X, mlp, unsloth_dequantize)
            assert_same(a, A, _F(_C()), dt)
            assert_same(b, B, _F(_C()), dt)
            assert_same(c, C, _F(_C()), dt)

        # Benchmarking
        torch.cuda.synchronize()
        start = time.time()
        for _ in range(1000): mlp_dequantize(X, mlp, dequantize_fx)
        elapsed += time.time() - start
    return elapsed

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1568: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
import torch
import triton
import triton.language as tl

_HAS_JOIN = hasattr(tl, "join")


@triton.jit
def _mul_rn(a, b):
    """fp32 multiply, explicit round-to-nearest, NOT contractible into an FMA."""
    return tl.inline_asm_elementwise(
        asm="mul.rn.f32 $0, $1, $2;",
        constraints="=f,f,f",
        args=[a, b],
        dtype=tl.float32,
        is_pure=True,
        pack=1,
    )


@triton.jit
def _add_rn(a, b):
    """fp32 add, explicit round-to-nearest, NOT contractible into an FMA."""
    return tl.inline_asm_elementwise(
        asm="add.rn.f32 $0, $1, $2;",
        constraints="=f,f,f",
        args=[a, b],
        dtype=tl.float32,
        is_pure=True,
        pack=1,
    )


@triton.jit
def _nf4_lookup(q):
    """Branchless 4-bit code -> NF4 value. 4-level select tree, no memory access."""
    hi = q >= 8
    b2 = (q & 4) != 0
    b1 = (q & 2) != 0
    b0 = (q & 1) != 0

    n0 = tl.where(b1, tl.where(b0, -0.39491748809814453, -0.5250730514526367),
                      tl.where(b0, -0.6961928009986877, -1.0))
    n1 = tl.where(b1, tl.where(b0, 0.0, -0.09105003625154495),
                      tl.where(b0, -0.18477343022823334, -0.28444138169288635))
    neg = tl.where(b2, n1, n0)

    p0 = tl.where(b1, tl.where(b0, 0.33791524171829224, 0.24611230194568634),
                      tl.where(b0, 0.16093020141124725, 0.07958029955625534))
    p1 = tl.where(b1, tl.where(b0, 1.0, 0.7229568362236023),
                      tl.where(b0, 0.5626170039176941, 0.44070982933044434))
    pos = tl.where(b2, p1, p0)

    return tl.where(hi, pos, neg)


@triton.jit
def _dequant_kernel_join(
    packed_ptr, absmax1_ptr, code2_ptr, absmax2_ptr, offset_ptr, out_ptr,
    n_blocks, BLOCKS: tl.constexpr,
):
    pid = tl.program_id(0)
    offset = tl.load(offset_ptr)

    blk = pid * BLOCKS + tl.arange(0, BLOCKS)
    bmask = blk < n_blocks

    c1 = tl.load(absmax1_ptr + blk, mask=bmask, other=0).to(tl.int32)
    a1 = tl.load(code2_ptr + c1, mask=bmask, other=0.0)
    a2 = tl.load(absmax2_ptr + (blk >> 8), mask=bmask, other=0.0)
    absmax = _add_rn(_mul_rn(a1, a2), offset)                     # (B,)

    # each packed byte loaded EXACTLY ONCE, contiguous (B, 32)
    bcol = tl.arange(0, 32)[None, :]
    row = blk[:, None]
    bmask2 = bmask[:, None]
    packed = tl.load(packed_ptr + row * 32 + bcol, mask=bmask2, other=0).to(tl.int32)

    hi_q = (packed >> 4) & 0x0F        # even elements
    lo_q = packed & 0x0F               # odd elements

    scale = absmax[:, None]
    v_hi = _nf4_lookup(hi_q) * scale   # (B, 32)
    v_lo = _nf4_lookup(lo_q) * scale   # (B, 32)

    # join -> (B, 32, 2) last dim [hi, lo]; reshape -> (B, 64) element order
    vals = tl.reshape(tl.join(v_hi, v_lo), (BLOCKS, 64))

    col = tl.arange(0, 64)[None, :]
    tl.store(out_ptr + row * 64 + col,
             vals.to(out_ptr.dtype.element_ty),
             mask=bmask2)


@triton.jit
def _dequant_kernel_dup(
    packed_ptr, absmax1_ptr, code2_ptr, absmax2_ptr, offset_ptr, out_ptr,
    n_blocks, BLOCKS: tl.constexpr,
):
    """Fallback for Triton without tl.join: loads each byte twice."""
    pid = tl.program_id(0)
    offset = tl.load(offset_ptr)

    blk = pid * BLOCKS + tl.arange(0, BLOCKS)
    bmask = blk < n_blocks

    c1 = tl.load(absmax1_ptr + blk, mask=bmask, other=0).to(tl.int32)
    a1 = tl.load(code2_ptr + c1, mask=bmask, other=0.0)
    a2 = tl.load(absmax2_ptr + (blk >> 8), mask=bmask, other=0.0)
    absmax = _add_rn(_mul_rn(a1, a2), offset)

    col = tl.arange(0, 64)[None, :]
    row = blk[:, None]
    tmask = bmask[:, None]

    packed = tl.load(packed_ptr + row * 32 + (col >> 1), mask=tmask, other=0).to(tl.int32)
    q = tl.where((col & 1) == 0, (packed >> 4) & 0x0F, packed & 0x0F)
    vals = _nf4_lookup(q) * absmax[:, None]

    tl.store(out_ptr + row * 64 + col, vals.to(out_ptr.dtype.element_ty), mask=tmask)


_KERNEL = _dequant_kernel_join if _HAS_JOIN else _dequant_kernel_dup

_BLOCKS = 16
_NUM_WARPS = 1
_NUM_STAGES = 2


def _your_dequantize_nf4(weight, quant_state):
    shape = quant_state.shape
    n_blocks = (shape[0] * shape[1]) >> 6

    out = torch.empty(shape, dtype=quant_state.dtype, device=weight.device)

    grid = ((n_blocks + _BLOCKS - 1) // _BLOCKS,)
    _KERNEL[grid](
        weight,
        quant_state.absmax,          # uint8 codes
        quant_state.state2.code,     # fp32, 256-entry map
        quant_state.state2.absmax,   # fp32
        quant_state.offset,          # fp32 scalar tensor, stays on device
        out,
        n_blocks,
        BLOCKS=_BLOCKS,
        num_warps=_NUM_WARPS,
        num_stages=_NUM_STAGES,
    )
    return out


def your_dequantize_nf4(weight):
    return _your_dequantize_nf4(weight.weight.data, weight.weight.quant_state)


In [6]:

import torch
import torch.nn as nn
from bitsandbytes.nn import Linear4bit
from transformers.activations import ACT2FN
from transformers import set_seed
from unsloth.kernels.utils import fast_dequantize


def unsloth_dequantize(weight):
    return fast_dequantize(weight.weight, weight.weight.quant_state)


def bnb_Linear4bit(hd, m, dtype):
    return Linear4bit(hd, m, bias=None, compute_dtype=dtype,
                      compress_statistics=True, quant_type="nf4")


class MLP(nn.Module):
    def __init__(self, hd, m, dtype):
        super().__init__()
        self.gate_proj = bnb_Linear4bit(hd, m, dtype).to("cuda")
        self.up_proj = bnb_Linear4bit(hd, m, dtype).to("cuda")
        self.down_proj = bnb_Linear4bit(m, hd, dtype).to("cuda")
        for p in (self.gate_proj, self.up_proj, self.down_proj):
            p.weight.quant_state.dtype = dtype
        self.act_fn = ACT2FN["silu"]

    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))


def mlp_forward(X, mlp, fx):
    up = X @ fx(mlp.up_proj).t()
    gate = X @ fx(mlp.gate_proj).t()
    return (mlp.act_fn(gate) * up) @ fx(mlp.down_proj).t()


def ulp_diff(a, b):
    """Difference in units-in-last-place for fp16/bf16 tensors."""
    ia = a.view(torch.int16).to(torch.int32)
    ib = b.view(torch.int16).to(torch.int32)
    ia = torch.where(ia < 0, 0x8000 - ia, ia)
    ib = torch.where(ib < 0, 0x8000 - ib, ib)
    return (ia - ib).abs()


# Same config as the failing case: (2, 3333, 2048, 8192, 3407, fp16)
BSZ, QLEN, HD, M, SEED, DT = 2, 3333, 2048, 8192, 3407, torch.float16

set_seed(SEED)
torch.set_default_dtype(torch.float32)
mlp = MLP(HD, M, DT)
X = torch.randn((BSZ, QLEN, HD), device="cuda", dtype=DT)
torch.cuda.synchronize()

print("=" * 70)
print("1. DEQUANT OUTPUT: yours vs unsloth  (the real correctness gate)")
print("=" * 70)
all_exact = True
for name in ("up_proj", "gate_proj", "down_proj"):
    mod = getattr(mlp, name)
    mine = your_dequantize_nf4(mod)
    theirs = unsloth_dequantize(mod)
    exact = torch.equal(mine, theirs)
    all_exact &= exact
    d = (mine.float() - theirs.float()).abs()
    n_diff = (mine != theirs).sum().item()
    u = ulp_diff(mine, theirs)
    print(f"  {name:10} bit_exact={exact}  n_diff={n_diff:>8}/{mine.numel():<10} "
          f"max_abs={d.max().item():.3e}  max_ulp={u.max().item()}")

print(f"\n  => {'BIT-EXACT. Your kernel is correct.' if all_exact else 'NOT bit-exact, see ULPs above.'}")

print()
print("=" * 70)
print("2. FORWARD ASSERT: does UNSLOTH itself pass it?")
print("=" * 70)
ref = mlp(X)
for label, fx in (("unsloth", unsloth_dequantize), ("yours", your_dequantize_nf4)):
    out = mlp_forward(X, mlp, fx)
    d = (out.float() - ref.float()).abs()
    rel = d / ref.float().abs().clamp_min(1e-30)
    n_bad = ((d > 1e-5) & (rel > 1e-3)).sum().item()
    try:
        torch.testing.assert_close(out, ref, check_stride=True)
        verdict = "PASSES"
    except AssertionError:
        verdict = "FAILS"
    print(f"  {label:8} vs bnb fused fwd: {verdict:7} "
          f"n_bad={n_bad:>4}/{out.numel()}  max_abs={d.max().item():.3e}")

print()
print("  => If unsloth ALSO fails, the assert is matmul nondeterminism, not your kernel.")

print()
print("=" * 70)
print("3. RERUN DETERMINISM: is the failure even reproducible?")
print("=" * 70)
o1 = mlp_forward(X, mlp, your_dequantize_nf4)
o2 = mlp_forward(X, mlp, your_dequantize_nf4)
print(f"  same fn, two runs identical: {torch.equal(o1, o2)}")
print(f"  bnb fused, two runs identical: {torch.equal(mlp(X), mlp(X))}")


1. DEQUANT OUTPUT: yours vs unsloth  (the real correctness gate)
  up_proj    bit_exact=True  n_diff=       0/16777216   max_abs=0.000e+00  max_ulp=0
  gate_proj  bit_exact=True  n_diff=       0/16777216   max_abs=0.000e+00  max_ulp=0
  down_proj  bit_exact=True  n_diff=       0/16777216   max_abs=0.000e+00  max_ulp=0

  => BIT-EXACT. Your kernel is correct.

2. FORWARD ASSERT: does UNSLOTH itself pass it?
  unsloth  vs bnb fused fwd: PASSES  n_bad=   0/13651968  max_abs=0.000e+00
  yours    vs bnb fused fwd: PASSES  n_bad=   0/13651968  max_abs=0.000e+00

  => If unsloth ALSO fails, the assert is matmul nondeterminism, not your kernel.

3. RERUN DETERMINISM: is the failure even reproducible?
  same fn, two runs identical: True
  bnb fused, two runs identical: True


In [9]:
from unsloth.kernels.utils import fast_dequantize
import statistics
print(f"GPU: {torch.cuda.get_device_name(0)}\n")

ratios = []
for run in range(1, 31):
    mine   = test_dequantize(your_dequantize_nf4)
    theirs = test_dequantize(unsloth_dequantize)
    r = theirs / mine
    ratios.append(r)
    print(f"run {run}:  unsloth {theirs:6.3f}s   mine {mine:6.3f}s   speedup {r:5.3f}x")

med = statistics.median(ratios)
print(f"\nmedian speedup : {med:.3f}x   (min {min(ratios):.3f}x, max {max(ratios):.3f}x)")
print(f"target 1.15x   : {'PASS' if med >= 1.15 else 'FAIL'}")


GPU: Tesla T4

run 1:  unsloth  4.116s   mine  3.300s   speedup 1.247x
run 2:  unsloth  4.542s   mine  3.307s   speedup 1.374x
run 3:  unsloth  4.558s   mine  3.291s   speedup 1.385x
run 4:  unsloth  4.271s   mine  3.364s   speedup 1.270x
run 5:  unsloth  4.282s   mine  3.308s   speedup 1.294x
run 6:  unsloth  4.366s   mine  3.313s   speedup 1.318x
run 7:  unsloth  4.377s   mine  3.421s   speedup 1.280x
run 8:  unsloth  4.305s   mine  3.494s   speedup 1.232x
run 9:  unsloth  4.281s   mine  3.276s   speedup 1.307x
run 10:  unsloth  4.523s   mine  3.284s   speedup 1.377x
run 11:  unsloth  4.336s   mine  3.279s   speedup 1.322x
run 12:  unsloth  4.323s   mine  3.385s   speedup 1.277x
run 13:  unsloth  4.398s   mine  3.337s   speedup 1.318x
run 14:  unsloth  4.348s   mine  3.293s   speedup 1.320x
run 15:  unsloth  4.337s   mine  3.437s   speedup 1.262x
run 16:  unsloth  4.310s   mine  3.504s   speedup 1.230x
run 17:  unsloth  4.951s   mine  3.317s   speedup 1.493x
run 18:  unsloth  5.352s 